<a href="https://colab.research.google.com/github/EstebanUMBJ/Google-Colab-Projects/blob/main/Sklearn%20Tutorial%3A%20Binary%20Text%20Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Análisis y predicción de reseñas positivas y negativas

###1. Importación de librerías

In [5]:
import pandas as pd
import numpy as np

df_review = pd.read_csv('IMDB Dataset.csv', engine='python')
df_review

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


###2. Muestra de datos

In [6]:
#Contar número de negativos y positivos
df_review.value_counts(['sentiment'])

,count
sentiment,
negative,25000
positive,25000


In [7]:
#Selección de una muestra
df_positivo = df_review[df_review['sentiment'] == 'positive'] [:9000]
df_negativo = df_review[df_review['sentiment'] == 'negative'] [:1000]

df_review_des = pd.concat([df_positivo, df_negativo])
df_review_des.value_counts(['sentiment'])



,count
sentiment,
positive,9000
negative,1000


###3. Dataset desbalanceado

In [8]:
 #Disminuir las muestras para luego entrenarla
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler()

#Organizar la data para que quede balanceada
df_review_bal, df_review_bal['sentiment'] = rus.fit_resample(df_review_des[['review']], df_review_des['sentiment'])
df_review_bal.value_counts(['sentiment'])
#

,count
sentiment,
negative,1000
positive,1000


###4. Separar data para entrenar (train) y para testear (test)

In [9]:
#Importar módulo para asignar un % de la data balanceada para entrenar y el restante para testear
from sklearn.model_selection import train_test_split
train, test = train_test_split(df_review_bal, test_size = 0.33, random_state = 42)

train_x, train_y = train ['review'], train['sentiment']
test_x, test_y = test ['review'], test['sentiment']



###5. Text representation (Bag of words)


In [10]:
#CountVectorizer --> Frecuencia de una palabra que aparece en una oración

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
text = ['Amo escribir código en Python. Amo el código en Python',
        'Odio escribir código en Java. Odio el código en Java']

df = pd.DataFrame({'review': ['review1', 'review2'], 'text': text})
cv = CountVectorizer()
cv_matrix = cv.fit_transform(df['text'])
df_dtm = pd.DataFrame(cv_matrix.toarray(), index=df['review'].values, columns=cv.get_feature_names_out())
df_dtm


,amo,código,el,en,escribir,java,odio,python
review1,2,2,1,2,1,0,0,2
review2,0,2,1,2,1,2,2,0


In [11]:
#Term Frequency - Inverse Document Frequency (Tfidf) --> Relevancia que tiene una palabra dentro de una oración, pero que no esté muy repetida en otros reviews

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
text = ['Amo escribir código en Python. Amo el código en Python',
        'Odio escribir código en Java. Odio el código en Java']

df = pd.DataFrame({'review': ['review1', 'review2'], 'text': text})
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['text'])
df_dtm = pd.DataFrame(tfidf_matrix.toarray(), index=df['review'].values, columns=cv.get_feature_names_out())
df_dtm

,amo,código,el,en,escribir,java,odio,python
review1,0.553373,0.393729,0.196865,0.393729,0.196865,0.000000,0.000000,0.553373
review2,0.000000,0.393729,0.196865,0.393729,0.196865,0.553373,0.553373,0.000000


###6. Turning our text data into numerical vectors

In [23]:
#Transformar data de texto a numérica

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)

test_x_vector = tfidf.transform(test_x)

In [13]:
train_x_vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 114998 stored elements and shape (1340, 19879)>

###7. Model selection

#### Support Vector Machines (SVM)

In [21]:
from sklearn.svm import SVC

svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

SVC(kernel='linear')

In [24]:
#Transformar oración para probar el testeo

print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all'])))


['negative']
['positive']
['negative']


#### Decision Tree

In [25]:
from sklearn.tree import DecisionTreeClassifier

dec_tree = DecisionTreeClassifier()
dec_tree.fit(train_x_vector, train_y)

DecisionTreeClassifier()

#### Naive Bayes

In [26]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(train_x_vector.toarray(), train_y)


GaussianNB()

#### Logistic Regression

In [27]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(train_x_vector, train_y)

LogisticRegression()

### 8. Model evaluation

####Mean Accuracy



In [31]:
#Return the mean accuracy on the given test data and labels.

print(svc.score(test_x_vector, test_y))
print(dec_tree.score(test_x_vector, test_y))
print(gnb.score(test_x_vector.toarray(), test_y))
print(log_reg.score(test_x_vector, test_y))

0.8303030303030303
0.6818181818181818
0.6378787878787879
0.8196969696969697


####F1 Score

In [35]:
#F1 Score is the weighted average of Precision and Recall.
#Accuracy is used when the True Positives and True negatives are more important while F1-score is used when the False Negatives and False Positives are crucial.
#Also, F1 takes into account how the data is distributed, so it's useful when you have data with imbalance classes.
#F1 Score = 2*(Recall * Precision) / (Recall + Precision)

from sklearn.metrics import f1_score

f1_score(test_y, svc.predict(test_x_vector), average=None, labels=['positive', 'negative'])

array([0.83382789, 0.82662539])

####Classification report

In [36]:
#Build a text report showing the main classification metrics.

from sklearn.metrics import classification_report

print(classification_report(test_y, svc.predict(test_x_vector), labels=['positive', 'negative']))

              precision    recall  f1-score   support

    positive       0.83      0.84      0.83       335
    negative       0.83      0.82      0.83       325

    accuracy                           0.83       660
   macro avg       0.83      0.83      0.83       660
weighted avg       0.83      0.83      0.83       660



####Confusion Matrix

In [37]:
#A confusion matrix is a table with two rows and two columns that reports the number of false positives, false negatives, true positives, and true negatives

from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y, svc.predict(test_x_vector), labels=['positive', 'negative'])
conf_mat

array([[281,  54],
       [ 58, 267]])

####Tuning the model: GridSearchCV

In [41]:
from sklearn.model_selection import GridSearchCV
parameters = {'C': [1,4,8,16,32], 'kernel' : ['linear', 'rbf']}
svc = SVC()
svc_grid = GridSearchCV(svc, parameters, cv=5)
svc_grid.fit(train_x_vector, train_y)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [1, 4, 8, 16, 32], 'kernel': ['linear', 'rbf']})

In [42]:
print(svc_grid.best_params_)
print(svc_grid.best_estimator_)

{'C': 1, 'kernel': 'rbf'}
SVC(C=1)


In [43]:
svc_grid.best_score_

np.float64(0.8305970149253732)